# Amazon ML Challenge 2026 — Business Entity Resolution
### Two-stage blocking + LightGBM matcher · end-to-end Kaggle runner

This notebook reproduces the full pipeline: **data → normalisation → blocking → stage-1 filter → stage-2 matcher → assignment → submission files**.

| Step | What happens | Output |
|---|---|---|
| 0 | Locate inputs, install 3 small libraries | — |
| 1 | **Train** on a slice of the labelled training data, tune threshold on held-out S1 entities | `artifacts/` (models, lexicon, meta) |
| 2 | **Predict** on the full test set | `output/matching_results.tsv`, `output/candidate_pairs.tsv` |
| 3 | **Validate** with the official `validate_submission.py` | PASS / FAIL |
| 4 | **Zip everything** for download | `output.zip`, `artifacts.zip`, `<team>_submission.zip` |

**Before running**
1. *Add Input* → the challenge zip (private dataset) **and** the `ber-code` dataset (the `business_entity_resolution` folder).
2. *Settings* → **Internet ON** (only for `pip install`), accelerator **TPU VM** if available (used purely for its many CPU cores), otherwise GPU/None — the pipeline is CPU-bound.
3. *Run All*. No external data or APIs are used anywhere — compliant with the challenge rules.

---
## 0 · Setup
### 0.1 Locate the dataset, the code and the official validator
Paths are discovered automatically under `/kaggle/input` (the `__MACOSX` junk folder inside the zip is ignored).

In [ ]:
import glob, os, time, json

def find(pattern):
    hits = [p for p in glob.glob('/kaggle/input/**/' + pattern, recursive=True) if '__MACOSX' not in p]
    assert hits, f'could not find {pattern} under /kaggle/input - did you add both datasets?'
    return hits[0]

DATA      = os.path.dirname(os.path.dirname(find('train/train_source1.tsv')))  # .../student_resource/dataset
CODE_SRC  = os.path.dirname(os.path.dirname(find('src/pipeline.py')))         # .../business_entity_resolution
VALIDATOR = find('utils/validate_submission.py')
DOC_TMPL  = find('Documentation_template.md')

WORK = '/kaggle/working'
ART  = f'{WORK}/artifacts'   # trained models + metadata
OUT  = f'{WORK}/output'      # submission TSVs

print('dataset   :', DATA)
print('code      :', CODE_SRC)
print('validator :', VALIDATOR)
print('CPU cores :', os.cpu_count())

### 0.2 Copy code to a writable folder and install dependencies
Only `rapidfuzz`, `sparse_dot_topn` and `Unidecode` are missing from the Kaggle image (pandas / numpy / scikit-learn / LightGBM are preinstalled).

In [ ]:
!rm -rf {WORK}/ber && cp -r "{CODE_SRC}" {WORK}/ber
!pip install -q rapidfuzz==3.14.5 sparse_dot_topn==1.2.0 Unidecode==1.4.0
%cd {WORK}/ber
!ls src

---
## 1 · Train
Stages inside `src.pipeline train`:

1. **Slice** ~25 % of training S1 entities (whole states, so distractor density stays realistic) + all their true matches + nearby unmatched records.
2. **Transliteration table** learned from matched pairs (Indic-script names → Latin).
3. **Normalise** names & addresses (legal forms incl. typos, DBA / "formerly known as", domains, states, house numbers, street abbreviations).
4. **Blocking** per (country, state): sparse TF-IDF top-K on name char-3-grams and address tokens.
5. **Stage-1 LightGBM** on cheap features → threshold keeping 99.95 % of true pairs (≈1 candidate per record).
6. **Stage-2 LightGBM** on the full feature set (token lexicon, vocabulary, house-number logic, context).
7. **Threshold tuning** for macro F0.5 on a 20 % held-out set of S1 entities.

Watch for the line **`best validation F0.5 … at threshold …`** at the end.

In [ ]:
t0 = time.time()
!python -u -m src.pipeline train --data-dir "{DATA}" --work {ART} 2>&1 | grep -v Warning
print(f'train finished in {(time.time()-t0)/60:.1f} min')

### 1.1 Training summary

In [ ]:
meta = json.load(open(f'{ART}/meta.json'))
print(f"validation macro F0.5 : {meta['val_f05']:.5f}")
print(f"decision threshold    : {meta['threshold']:.3f}")
print(f"stage-1 threshold     : {meta['stage1_threshold']:.5f}")
print(f"stage-2 features      : {len(meta['features2'])}")
!ls -la {ART}

> **12 h session limit:** if time is tight, stop here, *Save Version* so `artifacts/` becomes a notebook output,
> then run section 2 in a new session with that output added as an input (set `ART` to its path).

---
## 2 · Predict on the test set
Runs country by country (India, US, **France** — unseen in training) to keep memory low:
parse → block → stage-1 filter (survivors are written to `candidate_pairs.tsv`) → stage-2 scoring →
each S2/S3 record is assigned to its best S1 if the probability clears the tuned threshold.

In [ ]:
t0 = time.time()
!python -u -m src.pipeline predict --data-dir "{DATA}" --work {ART} --out {OUT}
print(f'predict finished in {(time.time()-t0)/60:.1f} min')

### 2.1 Quick look at the predictions

In [ ]:
import pandas as pd
m = pd.read_csv(f'{OUT}/matching_results.tsv', sep='\t', dtype=str, keep_default_na=False)
n_match = m.matched_entity_ids.str.count(',') + (m.matched_entity_ids != '')
print('S1 rows               :', len(m))
print('predicted singletons  :', round((n_match == 0).mean(), 4), '(train prior ~0.056)')
print('mean matches per S1   :', round(n_match.mean(), 3))
m.head(10)

---
## 3 · Validate with the official checker
Must print **PASS** before uploading. `--check-ids` also verifies every ID exists in the test set.

In [ ]:
!python "{VALIDATOR}" --matching {OUT}/matching_results.tsv --candidate {OUT}/candidate_pairs.tsv --test-dir "{DATA}/test" --check-ids

---
## 4 · Zip everything for download
Three zips are written to `/kaggle/working` (right panel → *Output* → download):

| File | Contents | Use |
|---|---|---|
| `output.zip` | `matching_results.tsv`, `candidate_pairs.tsv` | unzip → upload `matching_results.tsv` to the leaderboard |
| `artifacts.zip` | trained models, lexicon, transliteration table, `meta.json` | re-run `predict` later without retraining |
| `<team>_submission.zip` | `output/` + `code/business_entity_resolution/` + `Documentation_template.md` | final challenge package |

Set `TEAM` below. Fill in `Documentation_template.md` (see `FINDINGS.md` in the repo) before the final upload.

In [ ]:
TEAM = 'team_name'   # <- change me

# 1) predictions only
!cd {WORK} && rm -f output.zip && zip -qr output.zip output
# 2) trained artifacts
!cd {WORK} && rm -f artifacts.zip && zip -qr artifacts.zip artifacts
# 3) final submission package in the required structure
PKG = f'{WORK}/pkg'
!rm -rf {PKG} && mkdir -p {PKG}/output {PKG}/code
!cp {OUT}/matching_results.tsv {OUT}/candidate_pairs.tsv {PKG}/output/
!cp -r {WORK}/ber {PKG}/code/business_entity_resolution && rm -rf {PKG}/code/business_entity_resolution/src/__pycache__
!cp "{DOC_TMPL}" {PKG}/
!cd {PKG} && rm -f {WORK}/{TEAM}_submission.zip && zip -qr {WORK}/{TEAM}_submission.zip .
!rm -rf {PKG}
!ls -la {WORK}/*.zip

---
### Done ✅
* **Leaderboard:** download `output.zip`, unzip, upload `matching_results.tsv` on the portal.
* **Keep** `artifacts.zip` — lets you re-predict with a different threshold without retraining.
* **Final package:** download `<team>_submission.zip`.